
# Feed-Forward Neural Network — AQI Forecasting

**Tabular feed-forward neural network (FNN / MLP)** for the India AQI prediction. It follows the shared preprocessing pipeline and uses a temporal train/validation/test split to avoid leakage. It extends the simple baseline with a nonlinear learner over pollutants, lag/rolling AQI features, and city dummies.


In [13]:

import copy
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

from preprocessing import (
    load_and_prepare, impute, build_features,
    get_arrays, evaluate, save_results,
    CUTOFF_DATE,
)

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {DEVICE}')


device: cpu


## 1. Load data and create tabular features

In [14]:

df = load_and_prepare()
df = impute(df)
df, feature_cols = build_features(df)

print(f'rows after preprocessing: {len(df):,}')
print(f'number of tabular features: {len(feature_cols)}')
print('sample features:')
print(feature_cols[:15])


rows after preprocessing: 24,850
number of tabular features: 70
sample features:
['pm2.5', 'pm10', 'no', 'no2', 'nox', 'nh3', 'co', 'so2', 'o3', 'benzene', 'toluene', 'xylene', 'pm2.5_missing', 'pm10_missing', 'no_missing']



## 2. Temporal split

We keep the project's test split fixed at **2019-12-01** and carve a validation block out of the training period.


In [15]:

VAL_CUTOFF = pd.Timestamp('2019-10-01')

full_train_df = df[df['date'] < CUTOFF_DATE].copy().reset_index(drop=True)
test_df       = df[df['date'] >= CUTOFF_DATE].copy().reset_index(drop=True)

train_df = full_train_df[full_train_df['date'] < VAL_CUTOFF].copy().reset_index(drop=True)
val_df   = full_train_df[full_train_df['date'] >= VAL_CUTOFF].copy().reset_index(drop=True)

print(f'train rows: {len(train_df):,}')
print(f'val rows  : {len(val_df):,}')
print(f'test rows : {len(test_df):,}')
print(f'train date range: {train_df["date"].min().date()} to {train_df["date"].max().date()}')
print(f'val date range  : {val_df["date"].min().date()} to {val_df["date"].max().date()}')
print(f'test date range : {test_df["date"].min().date()} to {test_df["date"].max().date()}')


train rows: 18,431
val rows  : 1,319
test rows : 5,100
train date range: 2015-01-01 to 2019-09-30
val date range  : 2019-10-01 to 2019-11-30
test date range : 2019-12-01 to 2020-07-01


## 3. Build arrays and standardize inputs

In [16]:

X_train, X_val, y_train_log, y_val_raw, y_val_log = get_arrays(train_df, val_df, feature_cols)
_, X_test, _, y_test_raw, y_test_log = get_arrays(train_df, test_df, feature_cols)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train).astype(np.float32)
X_val_s   = scaler.transform(X_val).astype(np.float32)
X_test_s  = scaler.transform(X_test).astype(np.float32)

y_train_t = y_train_log.astype(np.float32).reshape(-1, 1)
y_val_t   = y_val_log.astype(np.float32).reshape(-1, 1)
y_test_t  = y_test_log.astype(np.float32).reshape(-1, 1)

print('scaled shapes:')
print('X_train:', X_train_s.shape, 'X_val:', X_val_s.shape, 'X_test:', X_test_s.shape)


scaled shapes:
X_train: (18431, 70) X_val: (1319, 70) X_test: (5100, 70)


## 4. Dataloaders

In [17]:

def make_loader(X, y, batch_size=256, shuffle=False):
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train_s, y_train_t, batch_size=256, shuffle=True)
val_loader   = make_loader(X_val_s, y_val_t, batch_size=512, shuffle=False)
test_loader  = make_loader(X_test_s, y_test_t, batch_size=512, shuffle=False)

len(train_loader), len(val_loader), len(test_loader)


(72, 3, 10)


## 5. Model definition


In [18]:

class AQIFeedForwardNN(nn.Module):
    def __init__(self, input_dim, hidden_dims=(128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.extend([
                nn.Linear(prev, h),
                nn.ReLU(),
                nn.BatchNorm1d(h),
                nn.Dropout(dropout),
            ])
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = AQIFeedForwardNN(input_dim=X_train_s.shape[1])
print(model)


AQIFeedForwardNN(
  (net): Sequential(
    (0): Linear(in_features=70, out_features=128, bias=True)
    (1): ReLU()
    (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=64, out_features=1, bias=True)
  )
)


## 6. Training utilities

In [19]:

def predict_log(model, loader, device='cpu'):
    model.eval()
    preds = []
    with torch.no_grad():
        for X_b, _ in loader:
            X_b = X_b.to(device)
            preds.append(model(X_b).cpu().numpy())
    return np.vstack(preds).reshape(-1)


def train_model(model, train_loader, val_loader, val_raw_aqi,
                epochs=150, lr=1e-3, weight_decay=1e-5,
                patience=15, device='cpu', verbose=True):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5
    )
    criterion = nn.MSELoss()

    best_state = None
    best_rmse = float('inf')
    wait = 0

    history = {
        'train_loss': [],
        'val_rmse': [],
    }

    for epoch in range(1, epochs + 1):
        model.train()
        batch_losses = []

        for X_b, y_b in train_loader:
            X_b = X_b.to(device)
            y_b = y_b.to(device)

            optimizer.zero_grad()
            pred = model(X_b)
            loss = criterion(pred, y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            batch_losses.append(loss.item())

        train_loss = float(np.mean(batch_losses))

        val_pred_log = predict_log(model, val_loader, device=device)
        val_pred_raw = np.clip(np.expm1(val_pred_log), 0, None)
        val_rmse = float(np.sqrt(mean_squared_error(val_raw_aqi, val_pred_raw)))

        history['train_loss'].append(train_loss)
        history['val_rmse'].append(val_rmse)

        scheduler.step(val_rmse)

        improved = val_rmse < best_rmse
        if improved:
            best_rmse = val_rmse
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if verbose and (epoch == 1 or epoch % 10 == 0):
            print(f'epoch {epoch:>3d} | train loss {train_loss:.4f} | val RMSE {val_rmse:.2f}')

        if wait >= patience:
            if verbose:
                print(f'early stopping at epoch {epoch}')
            break

    model.load_state_dict(best_state)
    return model, history, best_rmse


## 6b. Pre-tuned baseline evaluation

Train the **default architecture** (`hidden_dims=(128, 64)`, `dropout=0.2`, `lr=1e-3`) with no hyperparameter search and report the same metrics that are saved for the final tuned model.


In [20]:
# ── Pre-tuned baseline: default architecture, no search ─────────────────────
INDIA_AQI_BINS  = [0, 50, 100, 200, 300, 400, float('inf')]
INDIA_AQI_LABELS = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']

def print_metrics(y_true, y_pred, label='Model'):
    """Compute and pretty-print RMSE, MAE, R2, and per-category MAE."""
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    import pandas as pd

    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))

    # Per-category MAE using India AQI breakpoints
    tmp = pd.DataFrame({'actual': y_true, 'pred': y_pred})
    tmp['category'] = pd.cut(
        tmp['actual'],
        bins=INDIA_AQI_BINS,
        labels=INDIA_AQI_LABELS,
        right=True,
    )
    category_mae = (
        tmp.groupby('category', observed=True)
        .apply(lambda g: float(mean_absolute_error(g['actual'], g['pred'])))
        .reindex(INDIA_AQI_LABELS)
    )

    # ── Print results ────────────────────────────────────────────────────────
    sep = '─' * 52
    print(f'\n{sep}')
    print(f'  {label}')
    print(sep)
    print(f'  {"RMSE":<22} {rmse:>10.4f}')
    print(f'  {"MAE":<22} {mae:>10.4f}')
    print(f'  {"R²":<22} {r2:>10.4f}')
    print(f'  {"─" * 34}')
    print(f'  Per-category MAE (India AQI scale):')
    for cat in INDIA_AQI_LABELS:
        val = category_mae.get(cat, float('nan'))
        bar = int(val / 5) * '█' if not np.isnan(val) else ''
        print(f'  {cat:<16} {val:>8.4f}  {bar}')
    print(sep)

    return {
        'label': label,
        'RMSE': rmse, 'MAE': mae, 'R2': r2,
        **{f'MAE_{cat}': category_mae.get(cat, float('nan'))
           for cat in INDIA_AQI_LABELS},
    }


# Train default model --------------------------------------------------------
print('Training pre-tuned baseline (128, 64) …')
baseline_model = AQIFeedForwardNN(
    input_dim=X_train_s.shape[1],
    hidden_dims=(128, 64),
    dropout=0.2,
)
baseline_model, _hist_bl, best_val_rmse_bl = train_model(
    baseline_model,
    train_loader,
    val_loader,
    y_val_raw,
    epochs=150,
    lr=1e-3,
    weight_decay=1e-5,
    patience=15,
    device=DEVICE,
    verbose=True,
)
print(f'Pre-tuned best validation RMSE: {best_val_rmse_bl:.2f}')

# Evaluate on test set -------------------------------------------------------
bl_pred_log = predict_log(baseline_model, test_loader, device=DEVICE)
bl_pred_raw = np.clip(np.expm1(bl_pred_log), 0, None)

baseline_metrics = print_metrics(
    y_test_raw,
    bl_pred_raw,
    label='FeedForwardNN baseline (128, 64), dropout=0.2, lr=0.001',
)


Training pre-tuned baseline (128, 64) …
epoch   1 | train loss 23.9004 | val RMSE 222.83
epoch  10 | train loss 0.3396 | val RMSE 67.51
epoch  20 | train loss 0.1577 | val RMSE 48.25
epoch  30 | train loss 0.1316 | val RMSE 47.07
epoch  40 | train loss 0.1281 | val RMSE 45.97
early stopping at epoch 46
Pre-tuned best validation RMSE: 43.40

────────────────────────────────────────────────────
  FeedForwardNN baseline (128, 64), dropout=0.2, lr=0.001
────────────────────────────────────────────────────
  RMSE                      29.1438
  MAE                       16.4236
  R²                         0.8960
  ──────────────────────────────────
  Per-category MAE (India AQI scale):
  Good              12.5861  ██
  Satisfactory       9.8868  █
  Moderate          17.3752  ███
  Poor              31.9314  ██████
  Very Poor         26.8858  █████
  Severe           105.9024  █████████████████████
────────────────────────────────────────────────────


## 7. Hyperparameter search

In [ ]:

search_space = [
    {'hidden_dims': (128, 64), 'dropout': 0.20, 'lr': 1e-3, 'weight_decay': 1e-5},
    {'hidden_dims': (256, 128), 'dropout': 0.20, 'lr': 1e-3, 'weight_decay': 1e-5},
    {'hidden_dims': (128, 64, 32), 'dropout': 0.30, 'lr': 5e-4, 'weight_decay': 1e-4},
]

search_results = []

for i, cfg in enumerate(search_space, start=1):
    print(f'config {i}: {cfg}')
    candidate = AQIFeedForwardNN(
        input_dim=X_train_s.shape[1],
        hidden_dims=cfg['hidden_dims'],
        dropout=cfg['dropout'],
    )
    candidate, history, best_rmse = train_model(
        candidate,
        train_loader,
        val_loader,
        y_val_raw,
        epochs=120,
        lr=cfg['lr'],
        weight_decay=cfg['weight_decay'],
        patience=12,
        device=DEVICE,
        verbose=False,
    )
    record = dict(cfg)
    record['best_val_rmse'] = best_rmse
    record['epochs_ran'] = len(history['train_loss'])
    search_results.append(record)
    print(f"best val RMSE: {best_rmse:.2f} | epochs: {record['epochs_ran']}")

search_df = pd.DataFrame(search_results).sort_values('best_val_rmse').reset_index(drop=True)
search_df


SyntaxError: unterminated f-string literal (detected at line 10) (322983568.py, line 10)

## 8. Train selected configuration and inspect learning curves

In [ ]:

best_cfg = search_df.iloc[0].to_dict()
print('selected config:')
print(best_cfg)

best_model = AQIFeedForwardNN(
    input_dim=X_train_s.shape[1],
    hidden_dims=tuple(best_cfg['hidden_dims']),
    dropout=float(best_cfg['dropout']),
)

best_model, history, best_val_rmse = train_model(
    best_model,
    train_loader,
    val_loader,
    y_val_raw,
    epochs=150,
    lr=float(best_cfg['lr']),
    weight_decay=float(best_cfg['weight_decay']),
    patience=15,
    device=DEVICE,
    verbose=True,
)

print(f'best validation RMSE: {best_val_rmse:.2f}')


In [ ]:

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history['train_loss'], label='train loss')
ax.set_title('Training loss by epoch')
ax.set_xlabel('epoch')
ax.set_ylabel('MSE loss on log1p(AQI)')
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history['val_rmse'], label='validation RMSE')
ax.set_title('Validation RMSE by epoch')
ax.set_xlabel('epoch')
ax.set_ylabel('RMSE on original AQI scale')
ax.legend()
plt.tight_layout()
plt.show()


## 9. Evaluate on the held-out test set

In [ ]:

test_pred_log = predict_log(best_model, test_loader, device=DEVICE)
test_pred_raw = np.clip(np.expm1(test_pred_log), 0, None)

label = f"FeedForwardNN {tuple(best_cfg['hidden_dims'])}, dropout={best_cfg['dropout']}, lr={best_cfg['lr']}"
result = evaluate(y_test_raw, test_pred_raw, label=label)
save_results(result, 'fnn_tabular')


## 10. Predicted vs actual on the test set

In [ ]:

plot_df = pd.DataFrame({
    'actual_aqi': y_test_raw,
    'predicted_aqi': test_pred_raw,
})

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(plot_df['actual_aqi'], plot_df['predicted_aqi'], alpha=0.25)
lims = [0, max(plot_df['actual_aqi'].max(), plot_df['predicted_aqi'].max())]
ax.plot(lims, lims, linestyle='--')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel('Actual AQI')
ax.set_ylabel('Predicted AQI')
ax.set_title('Feed-forward NN: actual vs predicted AQI')
plt.tight_layout()
plt.show()
